


# Stock Price Forecast: SNOW, MSFT, AMZN

12-month share price prediction based on the last 12 months of daily closing data from the Nasdaq.

**Method:** Holt-Winters Exponential Smoothing with multiplicative seasonality, with 80% and 95% confidence intervals.

In [ ]:
SELECT TICKER, DATE, VALUE AS CLOSE_PRICE
FROM COLM_DB.STRUCTURED.STOCK_PRICE_TIMESERIES
WHERE TICKER IN ('SNOW', 'MSFT', 'AMZN')
  AND VARIABLE_NAME = 'Post-Market Close'
  AND DATE >= DATEADD(MONTH, -12, CURRENT_DATE())
ORDER BY TICKER, DATE

## Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = raw_prices.to_pandas()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['TICKER', 'DATE'])

tickers = ['SNOW', 'MSFT', 'AMZN']
colors = {'SNOW': '#29B5E8', 'MSFT': '#00A4EF', 'AMZN': '#FF9900'}
ticker_data = {}
for t in tickers:
    ts = df[df['TICKER'] == t][['DATE', 'CLOSE_PRICE']].set_index('DATE')
    ts = ts.asfreq('B', method='ffill')
    ticker_data[t] = ts
    print(f"{t}: {len(ts)} trading days, {ts.index.min().date()} to {ts.index.max().date()}, latest close ${ts['CLOSE_PRICE'].iloc[-1]:.2f}")

## Historical Price Chart (Last 12 Months)

In [ ]:
import plotly.graph_objects as go
import streamlit as st

fig = go.Figure()
for t in tickers:
    ts = ticker_data[t]
    fig.add_trace(go.Scatter(x=ts.index, y=ts['CLOSE_PRICE'], name=t, line=dict(color=colors[t], width=2)))

fig.update_layout(title='Historical Closing Prices (Last 12 Months)', xaxis_title='Date', yaxis_title='Price ($)', template='plotly_white', height=500)
st.plotly_chart(fig)

## Forecasting (12-Month Horizon)

Weekly resampled data fed into **Holt-Winters Exponential Smoothing** (additive trend, multiplicative seasonality). Confidence intervals computed from expanding residual standard deviation.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

forecast_weeks = 52
forecasts = {}

for t in tickers:
    ts = ticker_data[t]['CLOSE_PRICE'].resample('W-FRI').last().dropna()
    seasonal_periods = min(13, len(ts) // 2)

    model = ExponentialSmoothing(
        ts,
        trend='add',
        seasonal='mul',
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    )
    fit = model.fit(optimized=True)

    pred = fit.forecast(forecast_weeks)
    residuals = fit.resid.dropna()
    std_resid = residuals.std()

    steps = np.arange(1, forecast_weeks + 1)
    expanding_std = std_resid * np.sqrt(steps)

    forecasts[t] = {
        'historical_weekly': ts,
        'forecast': pred,
        'ci_80_lower': pred - 1.28 * expanding_std,
        'ci_80_upper': pred + 1.28 * expanding_std,
        'ci_95_lower': pred - 1.96 * expanding_std,
        'ci_95_upper': pred + 1.96 * expanding_std,
        'latest_price': ts.iloc[-1],
        'forecast_end_price': pred.iloc[-1]
    }
    pct = (pred.iloc[-1] / ts.iloc[-1] - 1) * 100
    print(f"{t}: ${ts.iloc[-1]:.2f} -> ${pred.iloc[-1]:.2f} ({pct:+.1f}%) over 12 months")

## Forecast Charts with Confidence Intervals

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, subplot_titles=[f'{t} Share Price Forecast' for t in tickers], vertical_spacing=0.08, shared_xaxes=False)

for i, t in enumerate(tickers, 1):
    f = forecasts[t]
    color = colors[t]

    fig.add_trace(go.Scatter(x=f['historical_weekly'].index, y=f['historical_weekly'], name=f'{t} Actual', line=dict(color=color, width=2), showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=f['forecast'].index, y=f['forecast'], name=f'{t} Forecast', line=dict(color=color, width=2, dash='dot'), showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=f['ci_80_upper'].index.tolist() + f['ci_80_lower'].index.tolist()[::-1], y=f['ci_80_upper'].tolist() + f['ci_80_lower'].tolist()[::-1], fill='toself', fillcolor='rgba(100,100,100,0.15)', line=dict(width=0), name='80% CI', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=f['ci_95_upper'].index.tolist() + f['ci_95_lower'].index.tolist()[::-1], y=f['ci_95_upper'].tolist() + f['ci_95_lower'].tolist()[::-1], fill='toself', fillcolor='rgba(100,100,100,0.07)', line=dict(width=0), name='95% CI', showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text='Price ($)', row=i, col=1)

fig.update_layout(height=1000, template='plotly_white', title_text='12-Month Stock Price Forecasts with Confidence Intervals')
st.plotly_chart(fig)

## Forecast Summary Table

In [ ]:
summary = []
for t in tickers:
    f = forecasts[t]
    current = f['latest_price']
    target = f['forecast_end_price']
    summary.append({
        'Ticker': t,
        'Current Price': f'${current:.2f}',
        '3M Forecast': f'${f["forecast"].iloc[12]:.2f}',
        '6M Forecast': f'${f["forecast"].iloc[25]:.2f}',
        '12M Forecast': f'${target:.2f}',
        'Expected Return': f'{(target/current - 1)*100:+.1f}%',
        '95% CI Low': f'${f["ci_95_lower"].iloc[-1]:.2f}',
        '95% CI High': f'${f["ci_95_upper"].iloc[-1]:.2f}'
    })

summary_df = pd.DataFrame(summary)
summary_df

## Combined Normalised Forecast (Rebased to 100)

In [ ]:
fig = go.Figure()
for t in tickers:
    f = forecasts[t]
    base = f['historical_weekly'].iloc[0]
    hist_norm = f['historical_weekly'] / base * 100
    fc_norm = f['forecast'] / base * 100
    fig.add_trace(go.Scatter(x=hist_norm.index, y=hist_norm, name=f'{t} Actual', line=dict(color=colors[t], width=2)))
    fig.add_trace(go.Scatter(x=fc_norm.index, y=fc_norm, name=f'{t} Forecast', line=dict(color=colors[t], width=2, dash='dot')))

fig.add_hline(y=100, line_dash='dash', line_color='grey', opacity=0.5)
fig.update_layout(title='Normalised Performance (Rebased to 100)', xaxis_title='Date', yaxis_title='Normalised Price', template='plotly_white', height=500)
st.plotly_chart(fig)

---

**Disclaimer:** These forecasts are based on statistical extrapolation of historical price patterns and do not constitute financial advice. Past performance is not indicative of future results. Always consult a qualified financial advisor before making investment decisions.